In [0]:
# Hack for AutoML DBR 17+
%pip install "urllib3<2.0" --quiet
dbutils.library.restartPython()

In [0]:
%run ./00-ddl

In [0]:
%run ./01-imports

In [0]:
import os
import requests
import json
import time
import pyspark.sql.functions as F
from datetime import datetime

# Predictions
import mlflow.pyfunc
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import FloatType

In [0]:
# Variables
table_silver = f'{CATALOG}.{DATABASE_S}.pitch_data'

In [0]:
# Read in Data
df_pitches = (
  spark.read
  .format('delta')
  .option("skipChangeCommits", "true")  # Ignores updates/deletes
  .table(table_silver)
)

# Subset Features
df = (
    df_pitches
    .filter("call_description IN ('Called Strike', 'Ball')")
    .select(
        F.col("season"),
        F.col("official_date"),
        F.col("game_pk"),
        F.col("at_bat_index"),
        F.col("pitch_index"),
        F.col("pitch_start_speed"),
        F.col("pitch_break_vertical_induced"),
        F.col("pitch_break_horizontal"),
        F.col("pitch_spin_rate"),
        F.col("position_x"),
        F.col("position_z"),
        F.col("is_strike")
    )
)

# Drop rows with missing values
df = df.dropna()

In [0]:
print((df.count(), len(df.columns)))

In [0]:
df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{DATABASE_S}.strike_probability_train")

## Kick off AutoML

In [0]:
import mlflow

# 0) Resolve your workspace username and make a safe experiment path
username = spark.sql("SELECT current_user()").first()[0]
exp_path = f"/Users/{username}/called_strike_automl"   # /Users/<you> exists by default
print(exp_path)

mlflow.set_experiment(exp_path)

# 1) Load training data (Spark DF)
df = spark.read.table(f"{CATALOG}.{DATABASE_S}.strike_probability_train")

# 2) Run AutoML programmatically (UI equivalent)
from databricks.automl.classifier import Classifier
from databricks.automl.internal.context import ContextType

clf = Classifier(context_type=ContextType.DATABRICKS)
automl_run = clf.fit(
    dataset=df,
    target_col="is_strike",
    exclude_cols=["season","official_date","game_pk","at_bat_index","pitch_index"],
    metric="f1",
    timeout_minutes=20
)

best_run_id = automl_run.best_trial.mlflow_run_id
print("Best trial run_id:", best_run_id)

In [0]:
import mlflow

# 3) Register best model to Unity Catalog
#    Choose a UC model name: <catalog>.<schema>.<model_name>
uc_model_name = "mlb_demos.mlb_gumbo_silver.strike_probability_model"

mlflow.set_registry_uri("databricks-uc")  # IMPORTANT for UC registry

from mlflow import MlflowClient
client = MlflowClient()

# source_uri = f"runs:/{best_run_id}/model"  # AutoML logs the model under 'model' artifact
source_uri = f"runs:/e19320b72dd344e5986044746d423795/model"  # AutoML logs the model under 'model' artifact
mv = mlflow.register_model(model_uri=source_uri, name=uc_model_name)
print(f"Registered model version: {mv.version}")

# 4) Wait until the model version is READY, then alias as 'prod'
import time

while True:
    mv_status = client.get_model_version(uc_model_name, mv.version).status
    if mv_status == "READY":
        break
    print(f"Waiting for model version {mv.version} to be READY... (status={mv_status})")
    time.sleep(2)

# ✅ Set alias to 'prod'
client.set_registered_model_alias(uc_model_name, "prod", mv.version)

print(f"✅ Model {uc_model_name} version {mv.version} is READY and aliased to 'prod'.")


In [0]:
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
import pandas as pd

MODEL_NAME = "mlb_demos.mlb_gumbo_silver.strike_probability_model"
SRC_VERSION = "1"   # trained on DBR 17.3 LTS

# Load the existing sklearn model (v2)
loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{SRC_VERSION}")
print(f"Loaded model version {SRC_VERSION} from {MODEL_NAME}")

FEATURE_NAMES = [
    "pitch_start_speed",
    "pitch_break_vertical_induced",
    "pitch_break_horizontal",
    "pitch_spin_rate",
    "position_x",
    "position_z",
]

class SklearnProbaWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model, feature_names):
        self.model = model
        self.feature_names = feature_names

    def predict(self, context, model_input):
        if isinstance(model_input, pd.DataFrame):
            model_input = model_input[self.feature_names]
        else:
            model_input = pd.DataFrame(model_input, columns=self.feature_names)
        proba = self.model.predict_proba(model_input)
        return proba[:, 1]

# sample input + signature
sample_input = pd.DataFrame({
    "pitch_start_speed": [90.0],
    "pitch_break_vertical_induced": [10.0],
    "pitch_break_horizontal": [5.0],
    "pitch_spin_rate": [2200],
    "position_x": [1.5],
    "position_z": [2.0],
})
wrapped_model = SklearnProbaWrapper(loaded_model, FEATURE_NAMES)
signature = infer_signature(sample_input, wrapped_model.predict(None, sample_input))

# 🔒 Exact DBR 17.3 LTS versions from release notes
pip_requirements = [
    # --- Core runtime ---
    "mlflow==3.0.1",               # current Databricks MLflow (3.0.1 also fine)
    "cloudpickle==3.0.0",
    "numpy==2.1.3",
    "pandas==2.2.3",
    "pyarrow==19.0.1",
    "scipy==1.15.1",
    "scikit-learn==1.6.1",
    "threadpoolctl==3.5.0",
    "joblib==1.4.2",

    # --- AutoML / feature / holiday transformers ---
    "databricks-automl-runtime==0.2.21",
    "databricks-feature-engineering==0.12.1",
    "category-encoders==2.6.3",
    "holidays==0.54",

    # --- ML algorithms / utilities ---
    "lightgbm==4.6.0",
    "matplotlib==3.10.0",
    "lz4==4.3.2",
    "cffi==1.17.1",
    "defusedxml==0.7.1",
    "psutil==5.9.0",
]


with mlflow.start_run(run_name="strike_probability_proba_wrapper_dbr173"):
    mlflow.pyfunc.log_model(
        name="model",
        python_model=wrapped_model,
        signature=signature,
        input_example=sample_input,
        registered_model_name=MODEL_NAME,
        pip_requirements=pip_requirements,
    )

print("\n✅ Wrapped proba model registered with DBR 17.3 LTS library pins.")
print("Point your serving endpoint at the new version.")
